In [1]:
import copy
import math
import random
import string

In [79]:
def read_iob2_file_wPOS(path):    
    """
    Read provided Universal NER iob2 file
    
    :param path: path to read from
    :returns: list with sequences of words, NER labels and POS tags for each sentence
    """
    data = []
    current_words = []
    current_ner_tags = []
    current_pos_tag = []

    for line in open(path, encoding='utf-8'):
        line = line.strip()

        if line:
            if line[0] == '#':
                continue # skip comments
            tok = line.split(' ')
            #print(tok)
            current_words.append(tok[1])
            current_ner_tags.append(tok[2])
            current_pos_tag.append(tok[3])
        else:
            if current_words:  # skip empty lines
                data.append((current_words, current_ner_tags,current_pos_tag))
            current_words = []
            current_ner_tags = []
            current_pos_tag = []

    # check for last one
    if current_ner_tags != []:
        data.append((current_words, current_ner_tags,current_pos_tag))
    return data

# Exclude certain numerical expressions from being manipulated by typo injector as apparently according to paper it would "mess with tokens"
numerical_expressions = ['zero', 'one', 'two', 'three', 'four',
                        'five', 'six', 'seven', 'eight', 'nine',
                        'ten', 'eleven', 'twelve', 'thirteen',
                        'fifteen', 'twenty', 'thirty', 'forty',
                        'fifty', 'hundred', 'thousand',
                        'million', 'billion']
numbers = ['1', '2', '3', '4', '5', '6', '7', '8', '9', '0']

# default typo type distribution as outlined in paper i reference
typo_types = ['insertion', 'replacement', 'deletion', 'transposition']
typo_type_samp_dist = [0.1525, 0.2825, 0.2825, 0.2825]

# First tuple are horizontally adjacent characters on keyboard
# Second tuple says whether a key is a left-handed key or not
keyboard_info = {
    'Q': (('W'), True),
    'W': (('Q','E'), True),
    'E': (('W','R'), True),
    'R': (('E','T'), True),
    'T': (('R','Y'), True),
    'Y': (('T','U'), False),
    'U': (('Y','I'), False),
    'I': (('U','O'), False),
    'O': (('I','P'), False),
    'P': (('O'), False),
    'A': (('S'), True),
    'S': (('A','D'), True),
    'D': (('S','F'), True),
    'F': (('D','G'), True),
    'G': (('F','H'), True),
    'H': (('G','J'), False),
    'J': (('H','K'), False),
    'K': (('J','L'), False),
    'L': (('K'), False),
    'Z': (('X'), True),
    'X': (('Z','C'), True),
    'C': (('X','V'), True),
    'V': (('C','B'), True),
    'B': (('V','N'), False),
    'N': (('B','M'), False),
    'M': (('N'), False),
}

def toSpans(tags):
    # Converts a list of tags to a list of spans
    # in: ['B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'O']
    # out: {'7-9:ORG', '0-2:PER'}
    spans = set()
    for beg in range(len(tags)):
        if tags[beg][0] == 'B':
            end = beg
            for end in range(beg+1, len(tags)):
                if tags[end][0] != 'I':
                    break
            spans.add(str(beg) + '-' + str(end) + ':' + tags[beg][2:])
    return spans

def typo_entity_generation(data, num_typos=1, target_tag="PER", print_output=False):
    """Injects a certain number of typos into a certain type of entity
    Parameters
    ----------
    data : list 
        Pass the data returned from the parser function, should be a list ex. [([words],[NER tags],[POS tags]),.....]
    num_typos : int
        Number of typos to insert into each typo
    target_tag : string
        Type of tag/entity to insert typos into
    Returns
    -------
    typo_data : list 
        A list in the same format as the input, but, now with typos of a variety of forms, plus an additional list of ints appended to the end that count how many errors are in a particular word.
    -----
    """
    typo_data = list([list(x) for x in list(data)])

    # for every data point in our data, perform a typo
    for data_point_i, data_point in enumerate(typo_data):
        sentence = data_point[0]
        if len(data_point) < 4: # debug check
            data_point.append([0]*len(sentence))

        if print_output:
            print("><><><><><><><><><><><><><><><")
            print(f"[DEBUG] Sentence: {sentence}")

        if len(sentence) < 2:
            continue

        # Weight words based on square root of their length for typo sampling as per paper
        word_samp_dist = [math.sqrt(len(word)) for word in sentence]

        if print_output:
            print(f"[DEBUG] Word sampling distribution 1: {word_samp_dist}")

        # Remove words that have any number-related components from sampling to avoid token corruption
        # Remove words that are entirely non-ASCII characters
        # Additionally, remove one-character-long words from being sampled
        # Additionally, if we want to only inject errors into person entities, do so
        for i in range(len(sentence)):
            word = sentence[i]

            if not target_tag in data_point[1][i]:
                word_samp_dist[i] = 0
                continue

            for num in numbers:
                if num in word:
                    if print_output:
                        print(f"[DEBUG] {num} IN {word}")
                    word_samp_dist[i] = 0
                    continue

            hasAscii = False
            for char in word:
                if char in string.ascii_letters:
                    hasAscii = True
            if not hasAscii:
                if print_output:
                    print(f"[DEBUG] {word} has NO ASCII letters")
                word_samp_dist[i] = 0
                continue

            if len(word) < 2:
                word_samp_dist[i] = 0
                continue
            for exp in numerical_expressions:
                if word.lower() == numerical_expressions:
                    word_samp_dist[i] = 0
                    continue

        if print_output:
            print(f">>> Word sampling distribution: {word_samp_dist}")

        # Normalize word sampling dist
        if sum(word_samp_dist) == 0:
            if print_output:
                print(f"[DEBUG] {sentence} was not sampleable...")
            continue
        # norm_word_samp_dist = [float(float(i)/sum(word_samp_dist)) for i in word_samp_dist]

        for span in toSpans(data_point[1]):
            if target_tag in span:
                if print_output:
                    print("~~~~~~~~~~~~~~~~~~~~~~~~~")
                    print(span)
                start, stop = span.split(':')[0].split('-')
                span_samp_dist = [0] * len(word_samp_dist)
                for malik in range(int(start), int(stop)+1):
                    span_samp_dist[malik] = word_samp_dist[malik]
                if print_output:
                    print(f"Span sampling distribution: {span_samp_dist}")
                norm_span_samp_dist = [float(float(i)/sum(span_samp_dist)) for i in span_samp_dist]
            else:
                continue

            for _ in range(num_typos):
                # Sample a word from word sampling dist
                foundWord = False
                for _ in range(25):
                    sampled_word, sampled_word_i = list_sample(sentence, norm_span_samp_dist)
                    if len(sampled_word) > 1:
                        foundWord = True
                        break
                if not foundWord:
                    if print_output or True:
                        print(f"[DEBUG] Failed to sample word from: {sentence}")
                    continue

                if print_output:
                    print(f"Normalized span sampling distribution: {norm_span_samp_dist}")
                    print(f"Sampled word: {sampled_word} of length {len(sampled_word)}")

                # This code block makes sure that if the typo error is transposition, only characters that are typed with opposite hands are transposed
                # Creates a list of indexes that will later on be used to set the sampling probability distribution to zero for characters typed with the same hand
                # Additionally, if a word is typed with only one hand, the sampled word will never sample a transposition typo error
                isAllOneHand = True
                transposition_set_zero = []
                for i in range(len(sampled_word)-1):
                    cur = sampled_word[i]
                    nxt = sampled_word[i+1]

                    # Makes sure that both letters are ASCII letters to be transposed
                    if not (cur in string.ascii_letters) or not (nxt in string.ascii_letters):
                        transposition_set_zero.append(i)
                        continue

                    # Exclude the first letter of a word that is not commonly capitalized
                    if i == 0 and cur.isupper() == nxt.islower():
                        transposition_set_zero.append(i)
                        continue
                    a = keyboard_info.get(cur.upper())[1]
                    b = keyboard_info.get(nxt.upper())[1]
                    if a == None or b == None:
                        transposition_set_zero.append(i)
                    elif a == b:
                        transposition_set_zero.append(i)
                    else:
                        isAllOneHand = False

                if print_output:
                    print(f"transposition_set_zero: {transposition_set_zero}")

                # Sample a typo type, 
                if not isAllOneHand:
                    sampled_typo_type, _ = list_sample(typo_types, typo_type_samp_dist)
                else:
                    # If word is typed with one hand, exclude transposition typo type
                    sampled_typo_type, _ = list_sample(typo_types[:3], [float(i)/sum(typo_type_samp_dist[:3]) for i in typo_type_samp_dist])
                
                if print_output:
                    print(f"Typo type: {sampled_typo_type}")

                # >>> Create character sampling distribution
                char_samp_dist = [0]*len(sampled_word)
                # Never sample first character to perform typo, but linearly interpolate between 0.1 weight and 0.2 weight for the 2nd and last letter
                char_samp_dist[1] = 0.1
                char_samp_dist[-1] = 0.2
                interp_indices = range(2,len(sampled_word)-1)
                for i in interp_indices:
                    char_samp_dist[i] = 0.1+0.1*(i-1)/(len(interp_indices)+1)

                for i in range(len(sampled_word)):
                    # Exclude non-ascii characters
                    if not sampled_word[i] in string.ascii_letters:
                        char_samp_dist[i] = 0
                    if sampled_typo_type == "transposition":
                        # Allow transposition for first letter
                        if not (i == 0 and sampled_word[i].isupper() == sampled_word[i+1].islower()):
                            char_samp_dist[0] = 0.05
                        # Set previously calculated non-opposite hand character to zero probability
                        for j in transposition_set_zero:
                            char_samp_dist[j] = 0
                        char_samp_dist[-1] = 0

                if sum(char_samp_dist) == 0 and sampled_word[0] in string.ascii_letters:
                    char_samp_dist[0] = 0.05
                # Normalize character sampling distribution and sample
                norm_char_samp_dist = [float(float(i)/sum(char_samp_dist)) for i in char_samp_dist]
                sampled_char, sampled_char_i = list_sample(sampled_word, norm_char_samp_dist)

                if print_output:
                    print(f"Character sampling distribution: {norm_char_samp_dist}")

                # Perform typo
                if sampled_typo_type == "insertion":
                    # Check if the nearest ASCII char neighbor on either side is capitalized or not
                    left = is_neighbor_capital(sampled_word, sampled_char_i, -1)
                    right = is_neighbor_capital(sampled_word, sampled_char_i, 1)

                    # Truth table to capitalize or not based on surrounding ASCII chars
                    doCapitalize = truth_table(left, right)
                    if print_output:
                        print(f"{left} - {right} | {doCapitalize}")

                    # If truth table results in saying we should capitalize, then capitalize
                    if doCapitalize:
                        inserted_char = random.sample(string.ascii_uppercase, 1)[0]
                    else:
                        inserted_char = random.sample(string.ascii_lowercase, 1)[0]

                    # Insert character and update our results
                    temp = sampled_word[:sampled_char_i]
                    temp += inserted_char
                    temp += sampled_word[sampled_char_i:]
                if sampled_typo_type == "replacement":
                    temp = list(sampled_word)
                    # Keep capitalization consistent
                    if sampled_char.isupper():
                        replacement_char = random.sample(keyboard_info[sampled_char][0], 1)[0]
                    else:
                        replacement_char = random.sample(keyboard_info[sampled_char.upper()][0], 1)[0].lower()
                    temp[sampled_char_i] = replacement_char
                    temp = "".join(temp)
                if sampled_typo_type == "deletion":
                    # Substring to exclude chosen character
                    temp = sampled_word[:sampled_char_i] + sampled_word[sampled_char_i+1:]
                if sampled_typo_type == "transposition":
                    # Substrings to transpose
                    temp = sampled_word[:sampled_char_i] + sampled_word[sampled_char_i+1] + sampled_word[sampled_char_i] + sampled_word[sampled_char_i+2:]
                
                if print_output:
                    print(f"Result: {temp}")

                typo_data[data_point_i][0][sampled_word_i] = temp
                typo_data[data_point_i][3][sampled_word_i] += 1

    return typo_data

def list_sample(lst, samp_dist):
    '''
    Given a list lst, and a list samp_dist of the same length, choose a random item in lst based on the normalized probability inside samp_dist
    '''
    sample = random.random()
    sampled_word = None
    temp_cum = 0
    for i in range(len(samp_dist)):
        lim = samp_dist[i]
        temp_cum += lim
        if sample <= temp_cum:
            return(lst[i], i)
    return (lst[-1], len(lst)-1) # if all else fails

def is_neighbor_capital(word, index, direction):
    '''
    Finds the nearest ASCII neighbor to the left or right to the desired inserted space
    Direction is either -1 or 1 (left or right)
    If the nearest ASCII neighbor to the left is the first character, then return None
    '''
    # If inserted characters are inserted to the right of the chosen index, then the index itself is the first left neighbor 
    if direction == -1:
        diff = 0
    else:
        diff = 1
    neighbor = index + (direction * diff)

    # Loop to find the nearest neighbor, skipping non-ASCII
    while not (neighbor < 1 or neighbor > len(word)-1):
        if word[neighbor] in string.ascii_letters:
            if word[neighbor] in string.ascii_uppercase:
                return True
            else:
                return False
        else:
            diff += 1
        neighbor = index + (direction * diff)
    return None

def truth_table(left, right):
    # Mimics the truth table seen in "typos_truth_table.txt"
    if left == True:
        if right == False:
            return random.choice([True, False])
        return True
    if left == False:
        if right == True:
            return random.choice([True, False])
        return False
    if right == True:
        return True
    if right == False:
        return False
    return random.choice([True, False])


In [43]:

data_dev = read_iob2_file_wPOS(r"../data/test_conll.iob2")

In [44]:
typo_data = typo_entity_generation(list([data_dev[10]]), num_typos=1, target_tag='PER')

[2.449489742783178, 2.449489742783178, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3.0, 3.1622776601683795, 0, 0, 0, 0, 0, 0, 0, 2.23606797749979, 2.23606797749979, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
['Takuya', 'Takagi', 'scored', 'the', 'winner', 'in', 'the', '88th', 'minute', ',', 'rising', 'to', 'head', 'a', 'Hiroshige', 'Yanagimoto', 'cross', 'towards', 'the', 'Syrian', 'goal', 'which', 'goalkeeper', 'Salem', 'Bitar', 'appeared', 'to', 'have', 'covered', 'but', 'then', 'allowed', 'to', 'slip', 'into', 'the', 'net', '.']
14-16:PER
['H', 'i', 'r', 'o', 'a', 'h', 'i', 'g', 'e']
0-2:PER
['T', 'a', 'k', 'a', 'g', 'o']
23-25:PER
19-20:MISC
[[['Takuya', 'Takago', 'scored', 'the', 'winner', 'in', 'the', '88th', 'minute', ',', 'rising', 'to', 'head', 'a', 'Hiroahige', 'Yanagimoto', 'cross', 'towards', 'the', 'Syrian', 'goal', 'which', 'goalkeeper', 'Slem', 'Bitar', 'appeared', 'to', 'have', 'covered', 'but', 'then', 'allowed', 'to', 'slip', 'into', 'the', 'net', '.'], ['B-PER', 'I-PER', 'O', 'O

In [80]:
def create_iob2_file_wPOS(data, path):    
    """
    Create a modified Universal NER iob2 file provided data
    
    :param path: path to write to
    :param data: data to convert
    :returns: nothing
    """
    with open(path, 'w') as f:
        for data_point in data:
            for i, zipped in enumerate(zip(data_point[0],data_point[1],data_point[2],data_point[3])):
                f.write(f"{i+1} {zipped[0]} {zipped[1]} {zipped[2]} {zipped[3]}\n")
            f.write("\n")

for i in range(10):
    data_dev = read_iob2_file_wPOS(r"../data/test_conll.iob2")
    typo_data = typo_entity_generation(data_dev, num_typos=1, target_tag='PER', print_output=False)
    create_iob2_file_wPOS(typo_data, f"../TestSets/typos_entity/typos_entity_test_conll_{i+1}.iob2")